<a href="https://colab.research.google.com/github/Lilo-Denise/Personal-AI-Teaching-Assistant/blob/main/Personal_AI_Teaching_Assistant_revised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# === Cell 1: 挂载云盘与路径设置 ===
import os
from google.colab import drive

# 1. 挂载 Google Drive
drive.mount('/content/drive')

# 2. 定义永久存储路径
# 所有文件都会存在你的 Google Drive -> Personal_TA_Project 文件夹下
PROJECT_ROOT = "/content/drive/MyDrive/Personal_TA_Project"
DATA_PATH = os.path.join(PROJECT_ROOT, "uploaded_pdfs") # 存放原始PDF
DB_PATH = os.path.join(PROJECT_ROOT, "vector_db")       # 存放AI的记忆

# 3. 自动创建文件夹
os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(DB_PATH, exist_ok=True)

print(f"✅ 云盘挂载成功！")
print(f"📂 文件存储位置: {DATA_PATH}")
print(f"🧠 记忆存储位置: {DB_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 云盘挂载成功！
📂 文件存储位置: /content/drive/MyDrive/Personal_TA_Project/uploaded_pdfs
🧠 记忆存储位置: /content/drive/MyDrive/Personal_TA_Project/vector_db


In [2]:
# === Cell 2: 修复依赖 (含 numpy) ===

# 1. 系统级依赖（PDF -> image）
!apt-get -q install -y poppler-utils

# 2. 修复 numpy：强制重装一个干净版本，避免 binary incompatibility
!pip install -q --force-reinstall "numpy==2.1.3"

import numpy as np
print("✅ numpy version:", np.__version__)

# 3. 安装/固定 NLP 相关依赖
!pip install -q \
  "transformers==4.46.3" \
  "sentence-transformers==3.2.1" \
  "langchain==0.1.20" \
  "langchain-community==0.0.38" \
  "chromadb>=0.5.0" \
  fastembed \
  gradio pypdf pdf2image pillow protobuf accelerate bitsandbytes

print("✅ 环境修复并安装完成！")



Reading package lists...
Building dependency tree...
Reading state information...
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.1.20 requires numpy<2,>=1, but you have numpy 2.1.3 which is incompatible.
langchain-community 0.0.38 requires numpy<2,>=1, but you have numpy 2.1.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.1.3 which is incompatible.
xarray 2025.11.0 requires packaging>=24.1, but you have packaging 23.2 which is incompatible.
db-dtypes 1.4.4 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
✅ numpy version: 1.26.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.5/108.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61

In [ ]:
# === Cell 3: 定义多模态处理工具 ===
import os
from pdf2image import convert_from_path
from langchain.schema import Document
from langchain_community.document_loaders import PyPDFLoader
from PIL import Image
import torch

print("正在初始化视觉模型 (用于看图说话)...")

image_captioner = None
_device = 0 if torch.cuda.is_available() else -1

try:
    # 注意：把 import 也放进 try 里，这样 numpy 有问题也不会把整个 cell 弄挂
    from transformers import pipeline as hf_pipeline

    image_captioner = hf_pipeline(
        "image-to-text",
        model="Salesforce/blip-image-captioning-base",
        device=_device,
    )
    print("✅ 视觉模型加载完成。")
except Exception as e:
    print("⚠️ 视觉模型加载失败，将仅使用文字内容。错误信息:", e)
    image_captioner = None


def process_pdf_multimodal(file_path: str):
    """读取 PDF，提取文字 + 图片描述，返回 LangChain Document 列表。"""
    file_name = os.path.basename(file_path)
    print(f"正在深入分析文件: {file_name} ...")

    documents = []

    # 1. 把 PDF 每一页转成图片（用于看图说话）
    try:
        images = convert_from_path(file_path, fmt="png")
    except Exception as e:
        print("⚠️ PDF 转图片失败，仅使用文字内容。错误信息:", e)
        images = []

    # 2. 用 PyPDFLoader 读取每一页的文字
    loader = PyPDFLoader(file_path)
    text_pages = loader.load()

    max_pages = max(len(text_pages), len(images))

    for page_idx in range(max_pages):
        # 对齐文字和图片
        text_page = text_pages[page_idx] if page_idx < len(text_pages) else None
        image = images[page_idx] if page_idx < len(images) else None

        page_text = text_page.page_content if text_page is not None else ""

        # 3. 让模型“看图说话”（如果前面加载失败，就不会走这里）
        visual_text = ""
        if image is not None and image_captioner is not None:
            try:
                caption = image_captioner(image, max_new_tokens=50)[0]["generated_text"]
                visual_text = f"[Visual Description]: {caption}"
            except Exception as e:
                print(f"⚠️ 第 {page_idx+1} 页图片描述失败:", e)
                visual_text = "[Visual Description]: (image not clear)"

        # 4. 合并：原文 + 视觉描述
        combined_content = page_text
        if visual_text:
            combined_content = page_text + "\n\n" + visual_text

        doc = Document(
            page_content=combined_content,
            metadata={"source": file_name, "page": page_idx + 1},
        )
        documents.append(doc)

    if not documents:
        print("⚠️ 未能从该 PDF 中解析出有效内容。")

    print(f"✅ 已完成 {len(documents)} 页的多模态解析。")

    return documents

print("✅ 多模态工具准备就绪！")



正在初始化视觉模型 (用于看图说话)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ 视觉模型加载完成。
✅ 多模态工具准备就绪！


In [ ]:
# === Cell 4: 知识库管理（含永久存储） ===
import os
import shutil
from typing import List

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import FastEmbedEmbeddings

# 说明：
# - 这里使用 FastEmbedEmbeddings，本地模型、无需 API
# - 不依赖 transformers，因此不会再出现 transformers.modeling_layers 的错误

# 1. 初始化 Embeddings（把文字变成向量）
print("正在加载 Embedding 模型（FastEmbed，本地，无 API）...")
embeddings = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")

vector_db = None

# 2. 尝试加载已有的数据库
if os.path.exists(DB_PATH) and len(os.listdir(DB_PATH)) > 0:
    try:
        print(f"发现已有知识库，正在从 {DB_PATH} 加载...")
        vector_db = Chroma(
            persist_directory=DB_PATH,
            embedding_function=embeddings,
        )
        print("✅ 知识库加载完成。\n")
    except Exception as e:
        print("⚠️ 旧知识库加载失败，将重新初始化。错误信息:", e)
        vector_db = None
else:
    print("未发现已有知识库，初始化为空库。\n")


def _reset_vector_db():
    """删除旧向量库并重新初始化。"""
    global vector_db
    if os.path.exists(DB_PATH):
        shutil.rmtree(DB_PATH)
    os.makedirs(DB_PATH, exist_ok=True)
    vector_db = None


def _rebuild_vector_db_from_files():
    """当删除文件后，根据 DATA_PATH 下剩余的 PDF 重建知识库。"""
    global vector_db

    pdf_files = [
        os.path.join(DATA_PATH, f)
        for f in os.listdir(DATA_PATH)
        if f.lower().endswith(".pdf")
    ]

    if not pdf_files:
        _reset_vector_db()
        return

    all_docs = []
    for path in pdf_files:
        # 使用 Cell 3 中定义的多模态解析函数
        all_docs.extend(process_pdf_multimodal(path))

    if not all_docs:
        _reset_vector_db()
        return

    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    splits = splitter.split_documents(all_docs)

    vector_db = Chroma.from_documents(
        documents=splits,
        embedding=embeddings,
        persist_directory=DB_PATH,
    )
    vector_db.persist()


def ingest_pdfs(uploaded_files) -> str:
    """处理上传的 PDF：保存到 DATA_PATH，并写入向量数据库。"""
    global vector_db

    if not uploaded_files:
        return "⚠️ Please upload at least one PDF."

    if not isinstance(uploaded_files, list):
        uploaded_files = [uploaded_files]

    new_docs = []

    # A. 保存文件到 DATA_PATH，并做多模态解析
    for f in uploaded_files:
        # gradio File 组件传进来的是一个类似 tempfile 的对象
        src_path = getattr(f, "name", None)
        if src_path is None:
            continue

        file_name = os.path.basename(src_path)
        dst_path = os.path.join(DATA_PATH, file_name)

        os.makedirs(DATA_PATH, exist_ok=True)
        shutil.copy(src_path, dst_path)

        docs = process_pdf_multimodal(dst_path)
        new_docs.extend(docs)

    if not new_docs:
        return "❌ 文件解析失败，可能是空文件。"

    # B. 切分文本
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    splits = text_splitter.split_documents(new_docs)

    # C. 存入/更新数据库
    if vector_db is None:
        vector_db = Chroma.from_documents(
            documents=splits,
            embedding=embeddings,
            persist_directory=DB_PATH,
        )
    else:
        vector_db.add_documents(splits)

    # 持久化到磁盘
    vector_db.persist()

    return f"✅ 成功！已添加 {len(uploaded_files)} 个文件，记忆库已更新。"


def list_stored_files() -> List[str]:
    """列出已上传的 PDF 文件名（来自 DATA_PATH）。"""
    if not os.path.exists(DATA_PATH):
        return []
    files = [
        f for f in os.listdir(DATA_PATH)
        if os.path.isfile(os.path.join(DATA_PATH, f))
    ]
    return sorted(files)


def delete_files(file_names: List[str]) -> int:
    """删除选中的文件，并同步刷新向量库。返回删除数量。"""
    global vector_db

    if not file_names:
        return 0

    if isinstance(file_names, str):
        file_names = [file_names]

    removed = 0
    for name in file_names:
        path = os.path.join(DATA_PATH, name)
        if os.path.exists(path):
            os.remove(path)
            removed += 1

    # 删除后重建向量库
    if removed > 0:
        _rebuild_vector_db_from_files()

    return removed


print("✅ 存储管理模块准备就绪！")


正在加载 Embedding 模型（FastEmbed，本地，无 API）...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/706 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model_optimized.onnx:   0%|          | 0.00/66.5M [00:00<?, ?B/s]

发现已有知识库，正在从 /content/drive/MyDrive/Personal_TA_Project/vector_db 加载...
✅ 知识库加载完成。

✅ 存储管理模块准备就绪！


In [ ]:
# === Cell 5: 加载本地 Qwen2 模型（不使用 transformers.pipeline） ===
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2-1.5B-Instruct"

print(f"正在加载本地模型: {model_id}")

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",   # 自动用 GPU / CPU
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

device = next(model.parameters()).device
print("✅ 模型加载完成，运行设备:", device)


def local_llm(prompt: str,
              max_new_tokens: int = 512,
              temperature: float = 0.2,
              top_p: float = 0.95,
              repetition_penalty: float = 1.15) -> str:
    """
    本地 LLM 推理函数：输入一个 prompt，返回生成的文本。
    不依赖 transformers.pipeline，也不调用任何远程 API。
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text



正在加载本地模型: Qwen/Qwen2-1.5B-Instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ 模型加载完成，运行设备: cuda:0


In [ ]:
# === Cell 6: 最终版 UI + 本地 LLM 兼容 ===
import gradio as gr
import pandas as pd
import os

# --- 1. 问答核心逻辑（保持原有结构与 UI 逻辑） ---
def answer_question_styled(message, history):
    # 空知识库提示
    if (vector_db is None or
        hasattr(vector_db, "_collection") and vector_db._collection.count() == 0):
        return (
            "⚠️ Knowledge base empty.",
            "<div class='source-item' style='color:#ef4444'>Please upload files first.</div>",
        )

    # 从向量库检索
    retriever = vector_db.as_retriever(search_kwargs={"k": 4})
    relevant_docs = retriever.invoke(message)

    sources_html_parts = []
    context_str = ""

    for i, doc in enumerate(relevant_docs):
        s_name = doc.metadata.get("source", "Unknown")
        p_num = doc.metadata.get("page", "?")
        sources_html_parts.append(
            f"<div class='source-item'>"
            f"<span class='source-tag'>#{i+1}</span> "
            f"<span class='source-name'>{s_name}</span> "
            f"<span class='source-page'>Pg.{p_num}</span>"
            f"</div>"
        )
        context_str += (
            f"Content {i+1} (Source: {s_name}, Page: {p_num}):\n"
            f"{doc.page_content}\n\n"
        )

    # prompt 模板沿用你原来的结构（带 system / user / assistant）
    prompt = f"""<|system|>
You are a direct academic assistant.
Answer strictly based on context. No fluff.
Cite inline like [1].
Context:
{context_str}
<|end|>
<|user|>
{message}
<|end|>
<|assistant|>"""

    # 兼容两种 local_llm 写法：有 .invoke 就用 invoke；否则当函数调用
    llm_obj = globals().get("local_llm")
    if llm_obj is None:
        raise RuntimeError("local_llm is not defined in Cell 5.")

    if hasattr(llm_obj, "invoke"):
        response = llm_obj.invoke(prompt)
    else:
        response = llm_obj(prompt)

    # 清洗回复文本
    if isinstance(response, str):
        raw_text = response
    else:
        # 有些 LLM 返回的是带 .content 的对象
        raw_text = getattr(response, "content", str(response))

    clean_response = raw_text.split("<|assistant|>")[-1].strip()
    sources_html = "<div class='source-box'>" + "".join(sources_html_parts) + "</div>"

    return clean_response, sources_html


# --- 2. 辅助逻辑（接口保持不变，但内部接到新的 Cell 4 函数） ---
def get_file_list_df():
    if not os.path.exists(DATA_PATH):
        return pd.DataFrame(columns=["File Name"])
    files = [f for f in os.listdir(DATA_PATH) if f.endswith(".pdf")]
    return pd.DataFrame(files, columns=["File Name"])


def refresh_options():
    if not os.path.exists(DATA_PATH):
        return []
    return [f for f in os.listdir(DATA_PATH) if f.endswith(".pdf")]


def handle_upload(files):
    # 对接 Cell 4 中的 ingest_pdfs（你最新版本里已经有）
    msg = ingest_pdfs(files)
    return msg, get_file_list_df(), gr.update(choices=refresh_options())


def handle_delete(file_name):
    # 对接 Cell 4 中的 delete_files（你最新版本里已经有）
    if not file_name:
        msg = "Please choose a file to delete."
    else:
        removed = delete_files([file_name])
        if removed > 0:
            msg = "Deleted file and rebuilt the knowledge base."
        else:
            msg = "File not found."
    return msg, get_file_list_df(), gr.update(choices=refresh_options(), value=None)


# --- 3. 原始 CSS：保留你的 UI 设计 ---
fix_css = """
/* 1. 全局黑底白字 */
body, .gradio-container {
    background-color: #000000 !important;
    color: #ffffff !important;
    font-family: 'Helvetica Neue', Arial, sans-serif !important;
}

/* 2. 左侧大标题 */
.main-header {
    font-size: 42px !important;
    font-weight: 800 !important;
    background: linear-gradient(90deg, #a78bfa, #ffffff);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    margin-bottom: 20px !important;
    padding-left: 5px;
}

/* 3. Section 标题 */
.section-label, .section-label * {
    color: #a1a1aa !important;
    font-size: 12px !important;
    letter-spacing: 0.18em !important;
    font-weight: 600 !important;
}

/* 4. 上传/删除 区域的“盒子” */
.file-box .wrap, .file-box input, .file-box .label-wrap, .file-box .input-box {
    background-color: #09090b !important;
    border-radius: 8px !important;
    border: 1px solid #27272a !important;
    color: #e4e4e7 !important;
}

/* 状态条 */
.status-box textarea, .status-box input {
    background-color: #09090b !important;
    border-radius: 8px !important;
    border: 1px solid #27272a !important;
    color: #a1a1aa !important;
    font-size: 12px !important;
}

/* 表格容器 */
.table-box table {
    background-color: #000000 !important;
    border-radius: 8px !important;
    border: 1px solid #27272a !important;
}
.table-box th {
    background-color: #09090b !important;
    color: #a1a1aa !important;
    font-size: 11px !important;
    text-transform: uppercase;
    letter-spacing: 0.12em;
}
.table-box td {
    background-color: #020617 !important;
    color: #e4e4e7 !important;
    font-size: 12px !important;
}

/* 5. 按钮：紫色渐变 */
.purple-btn {
    background: linear-gradient(135deg, #8b5cf6 0%, #d8b4fe 100%) !important;
    border: none !important;
    color: #000000 !important;
    font-weight: 800 !important;
    border-radius: 8px !important;
}

/* 6. 聊天窗口 */
.chat-window {
    border-radius: 16px !important;
    border: 1px solid #27272a !important;
    background: radial-gradient(circle at top left, #111827 0, #020617 55%, #000 100%) !important;
    height: 600px !important;
}

/* 用户气泡 */
.message-row.user-row .message {
    background-color: #27272a !important;
    border: none !important;
}
.message-row.user-row .message,
.message-row.user-row .message span,
.message-row.user-row .message p {
    color: #ffffff !important;
}

/* 助手气泡 */
.message-row.bot-row .message {
    background-color: #000000 !important;
    border: 1px solid #4b5563 !important;
}
.message-row.bot-row .message,
.message-row.bot-row .message span,
.message-row.bot-row .message p,
.message-row.bot-row .message strong,
.message-row.bot-row .message code {
    color: #ffffff !important;
}

/* 7. Sources 样式（右下角引用列表） */
.source-item {
    background-color: #111111;
    border-left: 3px solid #8b5cf6;
    padding: 10px;
    margin-bottom: 8px;
    border-radius: 4px;
    font-size: 13px;
}
.source-tag { color: #8b5cf6; font-weight: bold; margin-right: 8px; }
.source-name { color: white; font-weight: 500; }
.source-page { color: #71717a; float: right; }
"""


# --- 4. 界面构建：保持你原来的布局和文案 ---
with gr.Blocks(css=fix_css, theme=gr.themes.Monochrome()) as demo:

    with gr.Row():
        # === 左侧 ===
        with gr.Column(scale=1):
            gr.HTML("<div class='main-header'>PERSONAL AI<br>LEARNING<br>ASSISTANT</div>")

            # UPLOAD
            gr.Markdown("UPLOAD MATERIALS", elem_classes=["section-label"])
            file_input = gr.File(
                file_count="multiple",
                file_types=[".pdf"],
                container=True,
                elem_classes=["file-box"],
            )

            # Status
            upload_status = gr.Textbox(
                show_label=False,
                placeholder="Status messages will appear here...",
                interactive=False,
                elem_classes=["status-box"],
                max_lines=1,
            )

            upload_btn = gr.Button("Process & Save", elem_classes=["purple-btn"])

            # LIBRARY
            gr.Markdown("KNOWLEDGE LIBRARY", elem_classes=["section-label"])
            file_table = gr.Dataframe(
                headers=["File Name"],
                value=get_file_list_df(),
                interactive=False,
                elem_classes=["table-box"],
            )

            # MANAGE
            gr.Markdown("MAINTENANCE", elem_classes=["section-label"])
            delete_dropdown = gr.Dropdown(
                label=None,
                choices=refresh_options(),
                interactive=True,
                container=True,
                elem_classes=["file-box"],
            )
            delete_btn = gr.Button("Delete File", elem_classes=["purple-btn"])

        # === 右侧 ===
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label=None,
                show_label=False,
                type="messages",
                avatar_images=(None, None),
                elem_classes=["chat-window"],
                bubble_full_width=True,
            )

            gr.Markdown("VERIFIED SOURCES", elem_classes=["section-label"])
            sources_output = gr.HTML(
                "<div style='color:#555; font-style:italic'>No citations yet.</div>"
            )

            gr.Markdown("YOUR QUERY", elem_classes=["section-label"])
            with gr.Row():
                msg_input = gr.Textbox(
                    show_label=False,
                    placeholder="Ask anything about your materials...",
                    container=False,
                    scale=6,
                    elem_classes=["status-box"],
                )
                submit_btn = gr.Button(
                    "Ask",
                    elem_classes=["purple-btn"],
                    scale=2,
                )

            clear_btn = gr.Button("Clear History", elem_classes=["purple-btn"])

    # --- 5. 事件绑定 ---
    def process_query(message, history):
        history = history or []
        history.append({"role": "user", "content": message})
        yield history, "Searching...", ""
        ai_text, sources_html = answer_question_styled(message, history)
        history.append({"role": "assistant", "content": ai_text})
        yield history, "", sources_html

    msg_input.submit(
        process_query, [msg_input, chatbot], [chatbot, msg_input, sources_output]
    )
    submit_btn.click(
        process_query, [msg_input, chatbot], [chatbot, msg_input, sources_output]
    )

    upload_btn.click(
        handle_upload,
        inputs=[file_input],
        outputs=[upload_status, file_table, delete_dropdown],
    )
    delete_btn.click(
        handle_delete,
        inputs=[delete_dropdown],
        outputs=[upload_status, file_table, delete_dropdown],
    )
    clear_btn.click(
        lambda: ([], "<div style='color:#555'>Cleared.</div>"),
        None,
        [chatbot, sources_output],
    )

demo.launch(debug=True, share=True)



/tmp/ipython-input-1659752105.py:228: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=fix_css, theme=gr.themes.Monochrome()) as demo:
/tmp/ipython-input-1659752105.py:228: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=fix_css, theme=gr.themes.Monochrome()) as demo:
/tmp/ipython-input-1659752105.py:277: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(
/tmp/ipython-input-1659752105.py:277: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://21295c68afdca90c58.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


INFO:backoff:Backing off send_request(...) for 0.4s (requests.exceptions.ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')))


正在深入分析文件: Week 8_ Computer Vision.pdf ...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


✅ 已完成 65 页的多模态解析。


/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  warn_deprecated(


正在深入分析文件: Week 2_ MNIST part 1.pdf ...
✅ 已完成 45 页的多模态解析。
正在深入分析文件: Week 3_ MNIST_part2.pdf ...
✅ 已完成 60 页的多模态解析。
正在深入分析文件: Week 4_ LM - RNN.pdf ...
✅ 已完成 40 页的多模态解析。
正在深入分析文件: Week 5_ Language Models_ Transformers.pdf ...
✅ 已完成 57 页的多模态解析。
正在深入分析文件: Week 8_ Computer Vision.pdf ...
✅ 已完成 65 页的多模态解析。
